In [1]:
from fractions import Fraction
from typing import Tuple, Union
import numpy as np
import pandas as pd
from sympy import Matrix
from IPython.display import display

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.float_format', '{:.15f}'.format)
pd.set_option('display.width', 300)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Lặp đơn giải hệ tuyến tính

Hệ phương trình có dạng: $X = AX + B$

**Điều kiện hội tụ:**
Chuẩn của ma trận lặp $A$ phải nhỏ hơn 1: $q = \|A\|_p < 1$.

**Đánh giá sai số:**
- Tiên nghiệm: $\displaystyle\|X_n-X^{*}\|_p \leq \dfrac{q^{n}}{1-q}\displaystyle\|X_1-X_0\|_p$
- Hậu nghiệm: $\displaystyle\|X_n-X^{*}\|_p \leq \dfrac{q}{1-q}\displaystyle\|X_n-X_{n-1}\|_p$

# Thuật toán
**Input:** 
- Ma trận $A_{n \times n}, B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$.

**Output:** Nghiệm xấp xỉ $X^*$.

1. Tính $q = \|A\|_p$ với $p=1$ hoặc $p=\infty$. Nếu $q \geq 1 \Rightarrow$ Dừng thuật toán
2. Khởi tạo giá trị ban đầu $X_0$, khởi tạo $k = 1$
3. Lặp: 
- Tính $X_{k} = AX_{k-1} + B$
- Tính $\Delta_k = \|X_{k} - X_{k-1}\|_p$ hoặc $\delta_k=\dfrac{\|X_{k}-X_{k-1}\|_p}{\|X_{k}\|_p}$

    Lặp cho đến khi $\Delta_k \leq \dfrac{1-q}{q}\varepsilon$ hoặc $\delta_k \leq \dfrac{1-q}{q}\eta$
4. In ra $X^*$ là nghiệm xấp xỉ của hệ phương trình


# Thuật toán đánh giá sai số tiên nghiệm (Lặp đơn)

**Input**: 
- Ma trận $A_{n \times n}$ và $B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$

**Output**: Nghiệm xấp xỉ $X^*$ của hệ phương trình

1. Tính $q = \|A\|_p$ với $p=1$ hoặc $p=\infty$. Nếu $q \ge 1 \Rightarrow$ Dừng thuật toán.
2. Khởi tạo giá trị ban đầu $X_0 = [0]_{n \times m}$. Tính $X_1 = A X_0 + B$. 
3. Đặt $\Delta_1 = \|X_1 - X_0\|_p$. Nếu $\Delta_1 = 0 \Rightarrow$ Trả về $X^* = X_0$ và dừng chương trình.
4. Thiết lập khoảng sai số yêu cầu $\delta$:
    - Nếu dùng sai số tuyệt đối: $\delta = \varepsilon \dfrac{1 - q}{q}$.
    - Nếu dùng sai số tương đối: $\delta = \eta \|X_1\|_p \dfrac{1 - q}{q}$.
5. Tính số bước lặp tiên nghiệm $N$:
    - Nếu $\Delta_1 \le \delta \Rightarrow$ Gán $N = 1$.
    - Trái lại $\Rightarrow$ Gán $N = \left\lceil \dfrac{\ln(\delta / \Delta_1)}{\ln(q)} \right\rceil$.
6. Lặp $k$ từ $2$ đến $N$:
    - Tính $X_k = A X_{k-1} + B$.
7. In ra $X^* = X_N$ là nghiệm xấp xỉ của hệ phương trình.

In [2]:
def input_matrix(filename, convert_fractions=False):
    """
    Reads a matrix from a text file and returns it as a NumPy array.
    Supports fractional entries if present.
    """
    matrix = []

    with open(filename, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue

            row = []
            for token in tokens:
                if '/' in token:
                    # convert fractions if any includes fraction sign
                    val = Fraction(token)
                    row.append(float(val) if convert_fractions else val)
                else:
                    # parse as float directly
                    row.append(float(token))

            matrix.append(row)
            
    dtype = float if convert_fractions else object
    return np.array(matrix, dtype=dtype)

In [3]:
def output_matrix(X: np.ndarray, precision: int = 15):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    # Render natively as a LaTeX matrix in Jupyter
    display(Matrix(np.round(X, precision)))

In [4]:
def calc_norms(A: np.ndarray):
    tmp = np.abs(A)
    n = A.shape[0]
    
    col_sums = np.sum(tmp, axis=0)
    row_sums = np.sum(tmp, axis=1)
    max_elem = np.max(tmp)
    norm_1 = np.linalg.norm(tmp, ord=1)
    norm_2 = np.linalg.norm(tmp, ord=2)
    norm_inf = np.linalg.norm(tmp, ord=np.inf)
    norm_max = n * max_elem
    
    col_sums_str = ", ".join([f"{x:g}" for x in col_sums])
    row_sums_str = ", ".join([f"{x:g}" for x in row_sums])
    
    print("--- CÁC CHUẨN CỦA MA TRẬN LẶP A ---")
    print(f"Chuẩn cột (Norm 1): ||A||_1 = max({col_sums_str}) = {norm_1}")
    print(f"Chuẩn 2 (Norm 2): ||A||_2 = {norm_2}")
    print(f"Chuẩn hàng (Norm vô cùng): ||A||_inf = max({row_sums_str}) = {norm_inf}")
    print(f"Chuẩn max: ||A||_max = {n} * {max_elem:g} = {norm_max}")
    
    return norm_1, norm_2, norm_inf, norm_max

In [5]:
def fixed_point_matrix_iteration(A, B, initial_values, q, eps, eta, norm, mode='posteriori', max_iterations=1000):
    """
    Perform fixed-point iteration for system of linear equations using x_new = Ax + B
    Supports both 'apriori' and 'posteriori' stopping criteria.
    """
    if q >= 1:
        raise ValueError(f"Hệ số co q = {q} >= 1, thuật toán có thể phân kỳ hoặc không đảm bảo hội tụ!")
        
    if (eps is None) == (eta is None):
        raise ValueError("Specify exactly one of eps (exact) or eta (relative)")
    
    if norm == 1:
        vec_norm = lambda x: np.sum(np.abs(x)) if x.ndim == 1 else np.linalg.norm(x, ord=1)
    elif norm == 2:
        vec_norm = lambda x: np.linalg.norm(x, ord=2)
    elif norm == np.inf or norm == -1 or norm == 'max':
        vec_norm = lambda x: np.max(np.abs(x)) if x.ndim == 1 else np.linalg.norm(x, ord=np.inf)
    else:
        raise ValueError("Norm không hợp lệ. Chọn 1, 2, np.inf hoặc 'max'.")
    
    is_matrix = True if np.array(initial_values).ndim == 2 and np.array(initial_values).shape[1] > 1 else False
    
    new_eps = (eps if eps is not None else eta) * (1-q) / q
    print(f"Chế độ lặp: {mode.upper()}")
    print(f"Ngưỡng sai số (new_eps): {new_eps:.12f}")

    if not is_matrix:
        n = len(initial_values)
        values = np.array(initial_values, dtype=float)
        results = [values.tolist() + [np.nan]]

        if mode == 'apriori':
            x1 = np.dot(A, values) + B
            diff_1 = vec_norm(x1 - values)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_err = eps if eps is not None else eta * vec_norm(x1)
                if target_err is None: target_err = 1e-6
                target_diff = target_err * (1 - q) / q
                
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"Số bước lặp tính theo công thức tiên nghiệm: N = {max_iterations}")

        for i in range(1, max_iterations + 1):
            new_values = np.dot(A, values) + B
            
            if eps is not None:
                total_diff = vec_norm(new_values - values)
            else:
                total_diff = vec_norm(new_values - values) / vec_norm(new_values)
            
            results.append(new_values.tolist() + [total_diff])
            values = new_values
            
            if mode == 'posteriori' and new_eps is not None and total_diff <= new_eps:
                break
                
        if mode == 'posteriori' and total_diff > new_eps:
            print(f"Cảnh báo: Thuật toán không hội tụ sau {max_iterations} vòng lặp!")
        elif mode == 'apriori':
            print(f"Hoàn thành {max_iterations} vòng lặp theo công thức tiên nghiệm.")
            
        columns = [f'x{j+1}' for j in range(n)] + ['total_diff']
        import pandas as pd
        df = pd.DataFrame(results, columns=columns)
        df.index.name = "k"
        df_T = df.T
        df_T.index.name = "X_k"
        return df_T, values.reshape(-1, 1)
        
    else:
        values = np.array(initial_values, dtype=float)
        
        print("X_0 =")
        output_matrix(values)
        
        if mode == 'apriori':
            X1 = np.dot(A, values) + B
            diff_1 = vec_norm(X1 - values)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_err = eps if eps is not None else eta * vec_norm(X1)
                if target_err is None: target_err = 1e-6
                target_diff = target_err * (1 - q) / q
                
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"\n=> Mức sai số yêu cầu: ||X_n - X*|| <= {(target_diff * q / (1-q)):.4e}")
                print(f"=> Số bước lặp tính theo công thức tiên nghiệm: N = {max_iterations}")

        for i in range(1, max_iterations + 1):
            new_values = np.dot(A, values) + B
            
            if eps is not None:
                total_diff = vec_norm(new_values - values)
            else:
                total_diff = vec_norm(new_values - values) / vec_norm(new_values)
                
            print(f"\n--- Vòng lặp k = {i} ---")
            print(f"X_{i} =")
            output_matrix(new_values)
            err_str = f"{total_diff:.4e}" if total_diff < 1e-4 else f"{total_diff:.6f}"
            print(f"Sai số Delta_{i} = {err_str}")
            
            values = new_values
            
            if mode == 'posteriori' and new_eps is not None and total_diff <= new_eps:
                print(f"\n==> Hội tụ sau {i} bước lặp (sai số <= {new_eps:.4e}).")
                break
        else:
            if mode == 'posteriori':
                print(f"\n==> Đạt giới hạn lặp tối đa {max_iterations} bước.")
            else:
                print(f"\n==> Đã hoàn thành {max_iterations} bước lặp theo công thức tiên nghiệm.")
            
        return None, values

def solve_fixed_point_system(A, B, eps=1e-6, eta=None, norm=1, mode='posteriori', max_iterations=1000):
    if norm == 1:
        q = np.linalg.norm(np.abs(A), ord=1)
    elif norm == 2:
        q = np.linalg.norm(np.abs(A), ord=2)
    elif norm == np.inf or norm == -1 or norm == 'max':
        q = np.linalg.norm(np.abs(A), ord=np.inf)
    else:
        q = A.shape[0] * np.max(np.abs(A)) 
        
    print(f"\n[THIẾT LẬP] Sử dụng chuẩn {norm} => Tự động tính hệ số co q = {q:.6f}")
    
    if B.ndim == 1 or B.shape[1] == 1:
        B_mat = B.flatten()
        initial_guess = np.zeros_like(B_mat)
        df_history, X_final = fixed_point_matrix_iteration(A, B_mat, initial_guess, q, eps, eta, norm, mode, max_iterations)
        display(df_history)
        print("\nNghiệm gần đúng:")
        output_matrix(X_final)
        return X_final
    else:
        B_mat = B
        initial_guess = np.zeros((A.shape[1], B_mat.shape[1]))
        print("\n--- GIẢI CHO HỆ VỚI VẾ PHẢI LÀ MA TRẬN ---")
        _, X_final = fixed_point_matrix_iteration(A, B_mat, initial_guess, q, eps, eta, norm, mode, max_iterations)
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm X =")
        output_matrix(X_final)
        return X_final


# Kết quả

In [6]:
#Original matrix Ax=B
A = input_matrix('FXP_input_A2.txt', convert_fractions=True)
B = input_matrix('FXP_input_B2.txt', convert_fractions=True) # remove .flatten() to support multi-column B

print("\nMa trận A:")
output_matrix(A)
norm_1, norm_2, norm_inf, norm_max = calc_norms(A)
print("\nMa trận B:")
output_matrix(B)


Ma trận A:


Matrix([
[ 0.1,  0.2, -0.1, 0.0],
[-0.2,  0.1,  0.1, 0.2],
[ 0.1, -0.1,  0.3, 0.1],
[ 0.0,  0.2, -0.1, 0.4]])

--- CÁC CHUẨN CỦA MA TRẬN LẶP A ---
Chuẩn cột (Norm 1): ||A||_1 = max(0.4, 0.6, 0.6, 0.7) = 0.7000000000000001
Chuẩn 2 (Norm 2): ||A||_2 = 0.6066224063332756
Chuẩn hàng (Norm vô cùng): ||A||_inf = max(0.4, 0.6, 0.6, 0.7) = 0.7000000000000001
Chuẩn max: ||A||_max = 4 * 0.4 = 1.6

Ma trận B:


Matrix([
[ 1.0, -2.0,  0.5],
[ 3.0,  1.0, -1.0],
[-1.0,  4.0,  2.0],
[ 2.0, -1.0,  3.0]])

In [7]:
# Parameters
norm_type = 1 # CHỌN CHUẨN (1, 2, np.inf, 'max'). Hệ số q sẽ tự động được tính theo chuẩn này.
eps = 1e-6

# Giải hệ phương trình
X = solve_fixed_point_system(A, B, eps=eps, norm=norm_type)


[THIẾT LẬP] Sử dụng chuẩn 1 => Tự động tính hệ số co q = 0.700000

--- GIẢI CHO HỆ VỚI VẾ PHẢI LÀ MA TRẬN ---
Ngưỡng sai số (new_eps): 0.000000428571
X_0 =


Matrix([
[0.0, 0.0, 0.0],
[0.0, 0.0, 0.0],
[0.0, 0.0, 0.0],
[0.0, 0.0, 0.0]])


--- Vòng lặp k = 1 ---
X_1 =


Matrix([
[ 1.0, -2.0,  0.5],
[ 3.0,  1.0, -1.0],
[-1.0,  4.0,  2.0],
[ 2.0, -1.0,  3.0]])

Sai số Delta_1 = 8.000000

--- Vòng lặp k = 2 ---
X_2 =


...

Matrix([
[ 2.07144908748566, -2.37107460498444, 0.217516524822288],
[  3.8231055543416,  1.75741864929224, 0.227599998087951],
[-0.99683101049006,  4.85450881382249,   3.4975510554663],
[ 4.77384006120755, -1.88994519060667,  4.49294129888494]])

Sai số Delta_22 = 7.8695e-07

--- Vòng lặp k = 23 ---
X_23 =


Matrix([
[  2.07144912066589, -2.37107461202225, 0.217516546553189],
[  3.82310564912953,  1.75741862918703, 0.227600060167956],
[-0.996830943711857,  4.85450879965841,  3.49755109920182],
[  4.77384023640034, -1.88994522776647,  4.49294141362494]])

Sai số Delta_23 = 3.6994e-07

==> Hội tụ sau 23 bước lặp (sai số <= 4.2857e-07).

=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:
Hệ có nghiệm X =


Matrix([
[  2.07144912066589, -2.37107461202225, 0.217516546553189],
[  3.82310564912953,  1.75741862918703, 0.227600060167956],
[-0.996830943711857,  4.85450879965841,  3.49755109920182],
[  4.77384023640034, -1.88994522776647,  4.49294141362494]])